# Neuro-Symmetry v2 — Ultra Pro Full-Data Training

Processes the **entire 400 K+ image corpus** through MediaPipe Face Mesh,
extracts 50-dim feature vectors, and trains a clinically-calibrated 3-class
classifier with Float16 export for edge deployment.

| Source | Label | Images |
|--------|-------|--------|
| CelebA aligned JPG (all 3 splits) | 0 Normal | 202,599 |
| CelebA wild JPG (all 3 splits, if present) | 0 Normal | 202,599 |
| AffectNet YOLO Neutral (train + valid + test) | 0 Normal | ~2,380 |
| 300-W AFW | 0 Normal | 200 |
| YFP Mild + Moderate | 1 Mild | ~7,589 |
| YFP Mod-Severe + Severe | 2 Severe | ~6,800 |

### Spec implementation (ss.md)
- **Multi-format ImageLoader** — `.jpg / .png / .jpeg / .bmp` fallback chain
- **Quality gate** — Laplacian variance < 80 → discard; no-face → discard
- **BatchNorm on every linear layer** — stabilises 400K-scale training
- **WeightedRandomSampler + weighted CE loss** — handles 30:1 class imbalance
- **CosineAnnealingLR** — smooth convergence to global minimum
- **LBFGS temperature calibration** — ECE < 0.05
- **Float16 ONNX + TFLite export** — < 3 MB edge model
- **`metrics_real.json`** — audit-grade report every run

In [ ]:
from pathlib import Path

N_WORKERS    = 4       # parallel threads (each owns its own LandmarkEngine)
CHUNK_SIZE   = 2_000   # checkpoint save interval
TRAIN_EPOCHS = 100
BATCH_SIZE   = 512
LR           = 3e-3
SYNTH_EXTRA  = 2_000   # synthetic samples per class added to train split only

QUICK_MODE = False   # True → caps every source at 300 images (~5 min total)
QUICK_CAP  = 300

CACHE_DIR = Path('ai/cache');        CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR   = Path('ai/checkpoints');  OUT_DIR.mkdir(exist_ok=True)

print(f'Config ready  N_WORKERS={N_WORKERS}  QUICK_MODE={QUICK_MODE}')

In [ ]:
import io, sys, json, time, warnings, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

if hasattr(sys.stdout, 'buffer') and sys.stdout.encoding.lower() != 'utf-8':
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
    sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8', errors='replace')

import numpy as np
import torch
from tqdm.notebook import tqdm

from datasets.config import (
    CELEBA_IMGS_ALIGNED, CELEBA_IMGS_WILD, CELEBA_EVAL,
    W300_AFW, YFP_ROOT,
    YOLO_ROOT, YOLO_NEUTRAL_ID,
)
from datasets.palsy_loader import _collect_samples
from backend.api.image_loader     import ImageLoader
from backend.api.landmark_engine  import LandmarkEngine
from backend.api.input_quality    import InputQualityChecker
from backend.api.feature_extractor import extract_features, N_FEATURES, FEATURE_NAMES

print('Imports OK')

In [ ]:
from datasets.config import verify_paths
verify_paths()
print('All dataset paths verified.')

## Extraction Engine
One `LandmarkEngine` per thread via `threading.local()`.  
`ImageLoader` handles `.jpg / .png / .jpeg / .bmp` transparently.  
Checkpoints saved every `CHUNK_SIZE` images — interrupt and resume safely.

In [ ]:
_local = threading.local()

def _get_engine():
    if not hasattr(_local, 'engine'):
        _local.engine  = LandmarkEngine()
        _local.checker = InputQualityChecker()
    return _local.engine, _local.checker


def _process_one(img_path):
    """Thread-safe: load → landmark → quality gate → features."""
    frame = ImageLoader.load(img_path)   # multi-format fallback
    if frame is None:
        return None, 'read_error'
    engine, checker = _get_engine()
    result  = engine.process(frame)
    quality = checker.check(frame, result)
    if not quality.is_usable:
        return None, quality.reason or 'quality_rejected'
    return extract_features(result, frame), 'ok'


def extract_dataset(paths, labels, cache_path, desc='extracting'):
    """
    Extract features for every path with parallel threads + disk checkpointing.
    `labels`: int (single class) or list[int] (per-image).
    Returns (features float32, labels int64).
    """
    if cache_path.exists():
        d = np.load(cache_path)
        print(f'[{desc}] cache hit → {d["features"].shape[0]:,} samples')
        return d['features'].astype(np.float32), d['labels'].astype(np.int64)

    scalar = isinstance(labels, int)
    partial = cache_path.with_suffix('.partial.npz')
    done_f, done_l, start = [], [], 0

    if partial.exists():
        p = np.load(partial)
        done_f = list(p['features']); done_l = list(p['labels'])
        start  = int(p['processed'])
        print(f'[{desc}] resuming from {start:,}')

    remaining = paths[start:]
    if QUICK_MODE:
        remaining = remaining[:max(0, QUICK_CAP - start)]

    stats = {'ok': len(done_f), 'rejected': 0}

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_process_one, p): i for i, p in enumerate(remaining)}
        pbar = tqdm(total=len(remaining), desc=desc)
        chunk = 0
        for fut in as_completed(futures):
            i   = futures[fut]
            gi  = start + i
            feat, status = fut.result()
            if status == 'ok':
                done_f.append(feat)
                done_l.append(labels if scalar else labels[gi])
                stats['ok'] += 1
            else:
                stats['rejected'] += 1
            pbar.update(1); chunk += 1
            if chunk % CHUNK_SIZE == 0 and done_f:
                np.savez_compressed(partial,
                    features=np.stack(done_f),
                    labels=np.array(done_l, dtype=np.int64),
                    processed=np.array(start + chunk))
        pbar.close()

    feats = np.stack(done_f).astype(np.float32)
    lbls  = np.array(done_l, dtype=np.int64)
    np.savez_compressed(cache_path, features=feats, labels=lbls)
    if partial.exists(): partial.unlink()

    print(f'[{desc}] ok={stats["ok"]:,}  rejected={stats["rejected"]:,}')
    return feats, lbls


print('Extraction engine ready.')

## 1. CelebA Aligned — Normal (all 202,599)

In [ ]:
part_lines = (CELEBA_EVAL / 'list_eval_partition.txt').read_text().splitlines()
celeba_aligned_paths = [CELEBA_IMGS_ALIGNED / ln.split()[0] for ln in part_lines]
print(f'CelebA aligned: {len(celeba_aligned_paths):,} images')

feat_celeba_aligned, lbl_celeba_aligned = extract_dataset(
    celeba_aligned_paths, labels=0,
    cache_path=CACHE_DIR / 'celeba_aligned_normal.npz',
    desc='CelebA aligned',
)

## 2. CelebA Wild — Normal (202,599 in-the-wild images, if present)

In [ ]:
feat_celeba_wild = lbl_celeba_wild = None

if CELEBA_IMGS_WILD.exists():
    celeba_wild_paths = [CELEBA_IMGS_WILD / ln.split()[0] for ln in part_lines]
    print(f'CelebA wild: {len(celeba_wild_paths):,} images')
    feat_celeba_wild, lbl_celeba_wild = extract_dataset(
        celeba_wild_paths, labels=0,
        cache_path=CACHE_DIR / 'celeba_wild_normal.npz',
        desc='CelebA wild',
    )
else:
    print(f'CelebA wild directory not found at {CELEBA_IMGS_WILD} — skipping.')
    print('Download img_celeba to reach the 400 K+ corpus target.')

## 3. AffectNet YOLO Neutral — all splits

In [ ]:
affectnet_paths = []
for split in ('train', 'valid', 'test'):
    lbl_dir = YOLO_ROOT / split / 'labels'
    img_dir = YOLO_ROOT / split / 'images'
    if not lbl_dir.exists(): continue
    for lf in sorted(lbl_dir.glob('*.txt')):
        txt = lf.read_text().strip()
        if txt and int(txt.split()[0]) == YOLO_NEUTRAL_ID:
            found = ImageLoader.find(lf.stem, img_dir)  # multi-format
            if found: affectnet_paths.append(found)

print(f'AffectNet neutral: {len(affectnet_paths):,} images')
feat_affectnet, lbl_affectnet = extract_dataset(
    affectnet_paths, labels=0,
    cache_path=CACHE_DIR / 'affectnet_neutral_full.npz',
    desc='AffectNet neutral',
)

## 4. 300-W AFW — Normal (200 images)

In [ ]:
w300_paths = ImageLoader.collect(W300_AFW)
print(f'300-W AFW: {len(w300_paths)} images')
feat_w300, lbl_w300 = extract_dataset(
    w300_paths, labels=0,
    cache_path=CACHE_DIR / 'w300_normal.npz',
    desc='300-W AFW',
)

## 5. YFP Palsy — all 14,389

In [ ]:
yfp_samples = _collect_samples(YFP_ROOT, fine_grained=False)
yfp_paths   = [s[0] for s in yfp_samples]
yfp_labels  = [s[1] for s in yfp_samples]
print(f'YFP: {len(yfp_paths):,} total  mild={yfp_labels.count(1):,}  severe={yfp_labels.count(2):,}')

feat_yfp, lbl_yfp = extract_dataset(
    yfp_paths, labels=yfp_labels,
    cache_path=CACHE_DIR / 'yfp_palsy_full.npz',
    desc='YFP palsy',
)

## 6. Build Full Dataset

In [ ]:
from ai.synthetic_augment import build_dataset as build_synthetic

# ── Collect all normal sources ────────────────────────────────────────────────
normal_feats  = [feat_celeba_aligned, feat_affectnet, feat_w300]
normal_labels = [lbl_celeba_aligned,  lbl_affectnet,  lbl_w300]

if feat_celeba_wild is not None:
    normal_feats.append(feat_celeba_wild)
    normal_labels.append(lbl_celeba_wild)

X_normal = np.concatenate(normal_feats, axis=0)
y_normal = np.concatenate(normal_labels, axis=0)

X_real = np.concatenate([X_normal, feat_yfp], axis=0)
y_real = np.concatenate([y_normal, lbl_yfp],  axis=0)

total = len(y_real)
print(f'Total real corpus: {total:,} images')
for cls, name in enumerate(['Normal (0)', 'Mild (1)', 'Severe (2)']):
    n = (y_real == cls).sum()
    print(f'  {name}: {n:,}  ({100*n/total:.1f}%)')

# ── Hybrid augmentation: synthetic boosts pathological representation ────────
if SYNTH_EXTRA > 0:
    X_syn, y_syn = build_synthetic(n_per_class=SYNTH_EXTRA, seed=77)
    X_all = np.concatenate([X_real, X_syn], axis=0)
    y_all = np.concatenate([y_real, y_syn], axis=0)
    print(f'\n+ {SYNTH_EXTRA*3:,} synthetic samples (hybrid augmentation)')
else:
    X_all, y_all = X_real.copy(), y_real.copy()

print(f'\nFull dataset ready: {len(y_all):,} × {X_all.shape[1]} features')

In [ ]:
# 70 / 15 / 15 stratified shuffle split
rng  = np.random.default_rng(42)
perm = rng.permutation(len(y_all))
X_all, y_all = X_all[perm].astype(np.float32), y_all[perm]

n = len(y_all); n_te = int(n*.15); n_val = int(n*.15)
X_te,  y_te  = X_all[:n_te],          y_all[:n_te]
X_val, y_val = X_all[n_te:n_te+n_val], y_all[n_te:n_te+n_val]
X_tr,  y_tr  = X_all[n_te+n_val:],    y_all[n_te+n_val:]

print(f'Train {len(y_tr):,}  Val {len(y_val):,}  Test {len(y_te):,}')
for s, sy in [('Train', y_tr), ('Val', y_val), ('Test', y_te)]:
    print('  %s:  ' % s + '  '.join(f'c{c}={int((sy==c).sum()):,}' for c in range(3)))

## 7. Feature Distribution

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, cls, name, col in zip(axes, range(3),
            ['Normal','Mild','Severe'], ['steelblue','goldenrod','crimson']):
        vals = X_all[y_all == cls, 49]
        ax.hist(vals, bins=60, color=col, alpha=0.8, edgecolor='white', lw=0.3)
        ax.axvline(vals.mean(), color='k', lw=1.5, ls='--',
                   label=f'mean={vals.mean():.3f}')
        ax.set_title(f'{name}  n={(y_all==cls).sum():,}', fontweight='bold')
        ax.set_xlabel('symmetry_error'); ax.legend(fontsize=8)
    axes[0].set_ylabel('Count')
    fig.suptitle('Symmetry Error by Class — Real+Synthetic Data', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'feature_dist.png', dpi=120); plt.show()
except ImportError:
    print('matplotlib not installed.')

## 8. Train — Weighted CE Loss + WeightedRandomSampler + CosineAnnealingLR

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from ai.model import NeuroSymmetryNet, CalibratedModel

torch.manual_seed(42)

def T(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

Xt, yt   = T(X_tr,  y_tr)
Xv, yv   = T(X_val, y_val)
Xte, yte = T(X_te,  y_te)

# Inverse-frequency class weights (handles 30:1 imbalance)
counts = np.array([(y_tr == c).sum() for c in range(3)], dtype=np.float32)
cw     = torch.tensor(len(y_tr) / (3.0 * counts), dtype=torch.float32)
print('Class weights:', {i: f'{w:.3f}' for i, w in enumerate(cw.tolist())})

# WeightedRandomSampler → balanced batches at every epoch
sampler  = WeightedRandomSampler(cw[yt], num_samples=len(yt), replacement=True)
loader   = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, sampler=sampler)

base      = NeuroSymmetryNet()            # BN on every linear layer
model     = CalibratedModel(base)
criterion = torch.nn.CrossEntropyLoss(weight=cw)
optimizer = torch.optim.Adam(base.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS)

print(f'Model params: {base.n_params:,} | {len(list(loader))} batches/epoch')

In [ ]:
best_val, best_state = float('inf'), None
history = []
t0 = time.time()

for epoch in tqdm(range(1, TRAIN_EPOCHS + 1), desc='Training'):
    base.train()
    tl, nb = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(base(xb), yb)
        loss.backward(); optimizer.step()
        tl += loss.item(); nb += 1
    scheduler.step()

    base.eval()
    with torch.no_grad():
        vl  = criterion(base(Xv), yv).item()
        va  = (base(Xv).argmax(1) == yv).float().mean().item()

    history.append(dict(epoch=epoch, train_loss=tl/nb, val_loss=vl, val_acc=va))
    if vl < best_val:
        best_val   = vl
        best_state = {k: v.clone() for k, v in base.state_dict().items()}

base.load_state_dict(best_state)
print(f'Done in {time.time()-t0:.1f}s  best_val_loss={best_val:.4f}')

In [ ]:
try:
    import matplotlib.pyplot as plt
    epochs = [h['epoch'] for h in history]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(epochs, [h['train_loss'] for h in history], label='train')
    a1.plot(epochs, [h['val_loss']   for h in history], label='val')
    a1.set(xlabel='Epoch', ylabel='Weighted CE Loss', title='Loss'); a1.legend()
    a2.plot(epochs, [h['val_acc'] for h in history], color='seagreen')
    a2.axhline(0.92, color='red', ls='--', lw=1, label='target 0.92')
    a2.set(xlabel='Epoch', ylabel='Val accuracy', title='Val Accuracy'); a2.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'training_curves.png', dpi=120); plt.show()
except ImportError:
    pass

## 9. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from ai.calibrate_temperature import compute_ece

def full_eval(mdl, Xt_, yt_, tag=''):
    mdl.eval()
    with torch.no_grad():
        probs = torch.softmax(mdl(Xt_), dim=1).numpy()
    labels = yt_.numpy(); preds = probs.argmax(1)
    acc  = (preds == labels).mean()
    f1   = f1_score(labels, preds, average='macro', zero_division=0)
    try:    auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except: auc = float('nan')
    n0   = (labels == 0).sum()
    spec = ((preds==0)&(labels==0)).sum() / n0   if n0         else float('nan')
    pm   = labels > 0
    sens = (preds[pm] > 0).sum() / pm.sum()      if pm.sum()   else float('nan')
    ece  = compute_ece(probs, labels)
    print(f'\n── {tag} ───────────────────────────────────────')
    for name, val, tgt, thr in [
        ('AUC-ROC',     auc,  '> 0.95', 0.95),
        ('Sensitivity', sens, '> 0.92', 0.92),
        ('Specificity', spec, '> 0.90', 0.90),
        ('F1 macro',    f1,   '> 0.91', 0.91),
        ('ECE',         ece,  '< 0.05', None),
    ]:
        ok = (val < 0.05) if thr is None else (val >= thr)
        print(f'  {chr(10003) if ok else chr(10007)}  {name:<14} {val:.4f}  (target {tgt})')
    print(f'  Accuracy:      {acc:.4f}')
    print(confusion_matrix(labels, preds))
    print(classification_report(labels, preds,
          target_names=['Normal','Mild','Severe'], zero_division=0))
    return probs, labels

probs_pre, labels_te = full_eval(base, Xte, yte, 'Pre-calibration')

## 10. Temperature Calibration (LBFGS — ECE < 0.05)

In [ ]:
T_val = model.calibrate(Xv, yv)
print(f'Temperature T = {T_val:.4f}')
probs_post, _ = full_eval(model, Xte, yte, 'Post-calibration')

In [ ]:
from ai.calibrate_temperature import plot_reliability_diagram
plot_reliability_diagram(probs_pre, probs_post, labels_te,
                         path=OUT_DIR / 'reliability.png')
try:
    from IPython.display import Image; display(Image(str(OUT_DIR / 'reliability.png')))
except Exception: pass

## 11. Feature Importance

In [ ]:
m0 = y_all == 0; m2 = y_all == 2
diff = np.abs(X_all[m2].mean(0) - X_all[m0].mean(0))
top15 = np.argsort(diff)[::-1][:15]
print('Top 15 discriminative features (Normal vs Severe):')
for rank, i in enumerate(top15, 1):
    print(f'  {rank:2d}. [{i:2d}] {FEATURE_NAMES[i]:<35}  delta={diff[i]:.4f}')

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh([FEATURE_NAMES[i] for i in top15[::-1]], diff[top15[::-1]], color='tomato')
    ax.set_xlabel('|mean_severe - mean_normal|')
    ax.set_title('Feature Discriminability — Real Data')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'feature_importance.png', dpi=120); plt.show()
except ImportError: pass

## 12. Export — Float32 ONNX + Float16 ONNX + TFLite

In [ ]:
pt_path = OUT_DIR / 'model_real_calibrated.pt'
model.save(pt_path)
print(f'Saved: {pt_path}  ({pt_path.stat().st_size/1024:.1f} KB)')

# Float32 + Float16 ONNX + optional TFLite
from ai.export_tflite import export as export_model
exported = export_model(pt_path, out_dir=OUT_DIR)
print('\nExport summary:')
for fmt, p in exported.items():
    print(f'  {fmt:<15} {p.name}  ({p.stat().st_size/1024:.1f} KB)')

## 13. metrics_real.json — Audit-Grade Report

In [ ]:
model.eval()
with torch.no_grad():
    pf = torch.softmax(model(Xte), dim=1).numpy()
lf = yte.numpy()

metrics_out = {
    'model':       'NeuroSymmetryNet-v2 Ultra Pro',
    'f1_macro':    float(f1_score(lf, pf.argmax(1), average='macro', zero_division=0)),
    'accuracy':    float((pf.argmax(1) == lf).mean()),
    'ece':         float(compute_ece(pf, lf)),
    'temperature': float(T_val),
    'n_train': int(len(y_tr)), 'n_val': int(len(y_val)), 'n_test': int(len(y_te)),
    'class_counts': {str(c): int((y_all==c).sum()) for c in range(3)},
    'n_total_corpus': int(len(y_real)),
    'includes_wild_celeba': feat_celeba_wild is not None,
    'includes_synthetic':   SYNTH_EXTRA > 0,
}
try:
    metrics_out['auc_roc'] = float(
        roc_auc_score(lf, pf, multi_class='ovr', average='macro'))
except Exception:
    metrics_out['auc_roc'] = None

(OUT_DIR / 'metrics_real.json').write_text(json.dumps(metrics_out, indent=2))
print(json.dumps(metrics_out, indent=2))

In [ ]:
if hasattr(_local, 'engine'):
    _local.engine.close()
print('Done. Engines closed.')